# Module 02 — Supervised Learning
## 02-08: Naive Bayes for Text Classification

**Objective:** Implement bag-of-words and TF-IDF text representations and
multinomial Naive Bayes from scratch, revealing why the "naive" independence
assumption often works despite being false.

**Prerequisites:** 01-07 (Probability & Statistics — MLE, Bayes theorem),
01-08 (Information Theory — entropy, cross-entropy)

**GPU Required:** No (classical ML)

## Part 0 — Setup & Prerequisites

Naive Bayes is one of the oldest and most practical text classifiers. Despite
its "naive" assumption that all words are conditionally independent given the
class label — which is almost never true in real text — it achieves surprisingly
competitive accuracy, trains in a single pass over the data, and handles
high-dimensional sparse feature spaces (vocabulary sizes of 50,000+) with ease.

This notebook builds a complete text classification pipeline from scratch:
raw text → tokenisation → bag-of-words or TF-IDF vectorisation → Multinomial
Naive Bayes classifier. We evaluate on six categories from the **20 Newsgroups**
corpus (a classic NLP benchmark) and compare against scikit-learn's optimised
implementation.

The Bayes' theorem derivation here connects back to **01-07** (probability
foundations). The TF-IDF formula connects to **01-08** (information theory —
term frequency as a proxy for relevance).

In [ ]:
import sys
import re
import math
import time
import string
import warnings
from collections import Counter, defaultdict
from typing import Optional

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

warnings.filterwarnings("ignore")

print(f"Python : {sys.version.split()[0]}")
import torch
print(f"PyTorch: {torch.__version__}")
print(f"NumPy  : {np.__version__}")

In [ ]:
import random
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Configuration ─────────────────────────────────────────────────────────────
# Six semantically distinct newsgroup categories
CATEGORIES = [
    "sci.med",
    "sci.space",
    "rec.sport.hockey",
    "talk.politics.guns",
    "comp.graphics",
    "rec.motorcycles",
]
MIN_WORD_FREQ   = 2        # minimum document frequency to include in vocabulary
MAX_VOCAB_SIZE  = 10_000   # cap vocabulary at 10 000 most frequent terms
LAPLACE_ALPHA   = 1.0      # default Laplace smoothing coefficient
TEST_SIZE       = 0.20     # 80/20 train/test split
MAX_DOC_DISPLAY = 3        # number of sample documents to display in EDA

In [ ]:
# ── Fetch 20 Newsgroups (subset of 6 categories) ─────────────────────────────
# remove=("headers", "footers", "quotes") strips meta-information that would
# make the task artificially easy (e.g., newsgroup name in the header).
newsgroups_train = fetch_20newsgroups(
    subset="train",
    categories=CATEGORIES,
    remove=("headers", "footers", "quotes"),
    random_state=SEED,
)
newsgroups_test = fetch_20newsgroups(
    subset="test",
    categories=CATEGORIES,
    remove=("headers", "footers", "quotes"),
    random_state=SEED,
)

texts_train = newsgroups_train.data       # list of raw text strings
y_train_all = newsgroups_train.target     # integer class indices
texts_test  = newsgroups_test.data
y_test_all  = newsgroups_test.target

class_names = newsgroups_train.target_names  # human-readable category names

print(f"Train: {len(texts_train)} documents  |  classes: {len(class_names)}")
print(f"Test : {len(texts_test)}  documents")
print(f"Class distribution (train):")
for cls_idx, name in enumerate(class_names):
    count = int(np.sum(y_train_all == cls_idx))
    print(f"  [{cls_idx}] {name:<30}  {count:>4} docs")

In [ ]:
# ── EDA: document length distribution and sample documents ───────────────────
doc_lengths = [len(text.split()) for text in texts_train]
fig_eda, (ax_l, ax_b) = plt.subplots(1, 2, figsize=(13, 4))

ax_l.hist(doc_lengths, bins=50, color="steelblue", edgecolor="white", alpha=0.85)
ax_l.set_xlabel("Document length (words)")
ax_l.set_ylabel("Count")
ax_l.set_xlim(0, 800)
ax_l.set_title("Distribution of Document Lengths (Train)")
ax_l.axvline(np.median(doc_lengths), color="firebrick", linestyle="--",
             label=f"Median = {np.median(doc_lengths):.0f}")
ax_l.legend()
ax_l.grid(True, alpha=0.3)

class_counts = [int(np.sum(y_train_all == i)) for i in range(len(class_names))]
short_names  = [name.replace("talk.politics.", "pol.").replace("rec.", "")
                .replace("sci.", "sci.").replace("comp.", "") for name in class_names]
ax_b.barh(range(len(class_names)), class_counts, color="steelblue", alpha=0.85,
          edgecolor="black", linewidth=0.7)
for i, c in enumerate(class_counts):
    ax_b.text(c + 5, i, str(c), va="center", fontsize=9)
ax_b.set_yticks(range(len(class_names)))
ax_b.set_yticklabels(short_names, fontsize=9)
ax_b.set_xlabel("Document count")
ax_b.set_title("Class Distribution (Train)")
ax_b.grid(True, alpha=0.3, axis="x")

plt.suptitle("20 Newsgroups EDA (6 categories)", fontsize=12)
plt.tight_layout(); plt.show()

# Print a few sample documents
print(f"\nMean doc length : {np.mean(doc_lengths):.0f} words")
print(f"Median doc length: {np.median(doc_lengths):.0f} words")
print(f"\n--- Sample documents ---")
for cls_idx in [0, 2, 4]:
    idx = np.where(y_train_all == cls_idx)[0][0]
    snippet = texts_train[idx][:200].replace("\n", " ")
    print(f"[{class_names[cls_idx]}]\n  {snippet!r}\n")

---
## Part 1 — Text Representations & Naive Bayes Theory

### Bag-of-Words (BOW)

The simplest text representation: count how many times each word in the
vocabulary appears in the document, ignoring word order.

For a vocabulary $\mathcal{V} = \{w_1, w_2, \ldots, w_V\}$ and document $d$,
the BOW vector is:

$$\mathbf{x}_d = [\text{count}(w_1, d),\ \text{count}(w_2, d),\ \ldots,\ \text{count}(w_V, d)]$$

This creates a very sparse vector: a typical document uses <1% of the vocabulary.

### TF-IDF (Term Frequency – Inverse Document Frequency)

BOW counts do not distinguish between common words ("the", "and") and
informative words. TF-IDF down-weights ubiquitous terms:

$$\text{TF-IDF}(w, d) = \underbrace{\text{tf}(w,d)}_{\text{raw count}} \times \underbrace{\log\!\left(\frac{N+1}{\text{df}(w)+1}\right) + 1}_{\text{IDF (smooth)}}$$

where $N$ is the number of documents and $\text{df}(w)$ is the number of
documents containing word $w$. The smoothed IDF ensures words appearing in
every document still get a small non-zero weight.

In [ ]:
def tokenize(
    text: str,
    lowercase: bool = True,
    remove_punct: bool = True,
    min_length: int = 2,
) -> list[str]:
    """Tokenise a raw text string into a list of word tokens.

    Applies lowercasing, punctuation removal, and minimum token length
    filtering. Does not perform stopword removal — the Naive Bayes model
    learns to down-weight uninformative words automatically via the
    class-conditional log-probabilities.

    Args:
        text:         Raw text string to tokenise.
        lowercase:    If True, convert all characters to lowercase.
        remove_punct: If True, remove all punctuation characters.
        min_length:   Minimum token length to keep (shorter tokens dropped).

    Returns:
        List of string tokens (may be empty for very short or empty input).
    """
    if lowercase:
        text = text.lower()
    if remove_punct:
        text = re.sub(r"[^a-z0-9\s]", " ", text)
    tokens = text.split()
    return [tok for tok in tokens if len(tok) >= min_length]


# Quick unit test
sample = "Hello, World! This is a test of Tokenise-me 123."
tokens = tokenize(sample)
print(f"Input : {sample!r}")
print(f"Output: {tokens}")

In [ ]:
class BagOfWords:
    """Bag-of-words vectoriser that converts text documents to count matrices.

    Builds a vocabulary from the training corpus (filtered by minimum
    document frequency), then maps each document to a count vector.
    Unknown words at transform time are silently ignored.

    Attributes:
        min_df:      Minimum document frequency for a term to be included.
        max_features: If set, keep only the top max_features most frequent terms.
        vocab_:      Mapping of term to column index, built during fit().
        idx_to_term_: Reverse mapping of column index to term.
        n_docs_:     Number of training documents seen during fit().
    """

    def __init__(
        self,
        min_df: int = 2,
        max_features: Optional[int] = None,
    ) -> None:
        """Initialise BagOfWords hyperparameters.

        Args:
            min_df:       Minimum number of documents a term must appear in.
            max_features: Maximum vocabulary size (keep top-frequency terms).
        """
        self.min_df = min_df
        self.max_features = max_features
        self.vocab_: dict[str, int] = {}
        self.idx_to_term_: dict[int, str] = {}
        self.n_docs_: int = 0

    def fit(self, corpus: list[str]) -> "BagOfWords":
        """Build vocabulary from the training corpus.

        Tokenises each document, counts document frequency for each term,
        filters by min_df, and assigns column indices to surviving terms.

        Args:
            corpus: List of raw text strings (training documents).

        Returns:
            self (fitted vectoriser).
        """
        self.n_docs_ = len(corpus)
        doc_freq: Counter = Counter()
        for text in corpus:
            tokens = set(tokenize(text))   # unique per document for df count
            doc_freq.update(tokens)

        # Filter by minimum document frequency
        valid_terms = [(term, freq) for term, freq in doc_freq.items()
                       if freq >= self.min_df]
        # Sort by descending frequency, then alphabetically for ties
        valid_terms.sort(key=lambda x: (-x[1], x[0]))
        if self.max_features is not None:
            valid_terms = valid_terms[: self.max_features]

        self.vocab_ = {term: idx for idx, (term, _) in enumerate(valid_terms)}
        self.idx_to_term_ = {idx: term for term, idx in self.vocab_.items()}
        return self

    def transform(self, corpus: list[str]) -> np.ndarray:
        """Transform a list of documents into a BOW count matrix.

        Args:
            corpus: List of raw text strings to vectorise.

        Returns:
            Integer count matrix, shape (n_docs, vocab_size).
        """
        V = len(self.vocab_)
        X = np.zeros((len(corpus), V), dtype=np.float64)
        for i, text in enumerate(corpus):
            tokens = tokenize(text)
            for tok in tokens:
                if tok in self.vocab_:
                    X[i, self.vocab_[tok]] += 1.0
        return X

    def fit_transform(self, corpus: list[str]) -> np.ndarray:
        """Fit vocabulary and transform corpus in one step.

        Args:
            corpus: List of raw text strings.

        Returns:
            BOW count matrix, shape (n_docs, vocab_size).
        """
        return self.fit(corpus).transform(corpus)


# ── Fit BOW on training data ──────────────────────────────────────────────────
bow = BagOfWords(min_df=MIN_WORD_FREQ, max_features=MAX_VOCAB_SIZE)
X_bow_train = bow.fit_transform(texts_train)
X_bow_test  = bow.transform(texts_test)

V = len(bow.vocab_)
print(f"Vocabulary size : {V:,} terms")
print(f"BOW train shape : {X_bow_train.shape}")
print(f"BOW test  shape : {X_bow_test.shape}")
print(f"Sparsity (train): {(X_bow_train == 0).mean():.2%} of entries are zero")

# Show top-10 most frequent terms
term_freqs = X_bow_train.sum(axis=0)
top10_idx = np.argsort(term_freqs)[::-1][:10]
print(f"\nTop-10 most frequent terms in training set:")
for rank, idx in enumerate(top10_idx, 1):
    print(f"  {rank:2d}. {bow.idx_to_term_[idx]:<20} count={int(term_freqs[idx]):>8,}")

In [ ]:
class TFIDF:
    """TF-IDF vectoriser built from scratch.

    Uses the smoothed IDF formula to prevent zero-division:
        IDF(w) = log((N + 1) / (df(w) + 1)) + 1
    where N = number of training documents and df(w) = document frequency.

    Attributes:
        min_df:       Minimum document frequency filter.
        max_features: Maximum vocabulary size.
        vocab_:       Term-to-index mapping, set during fit().
        idf_:         IDF weight for each vocabulary term, shape (V,).
        n_docs_:      Number of training documents seen.
    """

    def __init__(
        self,
        min_df: int = 2,
        max_features: Optional[int] = None,
    ) -> None:
        """Initialise TFIDF hyperparameters.

        Args:
            min_df:       Minimum document frequency for a term to be kept.
            max_features: Vocabulary size cap (keep top-frequency terms).
        """
        self.min_df = min_df
        self.max_features = max_features
        self.vocab_: dict[str, int] = {}
        self.idf_: np.ndarray | None = None
        self.n_docs_: int = 0

    def fit(self, corpus: list[str]) -> "TFIDF":
        """Build vocabulary and compute IDF weights from training corpus.

        Args:
            corpus: List of raw text strings (training documents).

        Returns:
            self (fitted vectoriser).
        """
        self.n_docs_ = len(corpus)
        doc_freq: Counter = Counter()
        for text in corpus:
            tokens = set(tokenize(text))
            doc_freq.update(tokens)

        valid_terms = [(term, freq) for term, freq in doc_freq.items()
                       if freq >= self.min_df]
        valid_terms.sort(key=lambda x: (-x[1], x[0]))
        if self.max_features is not None:
            valid_terms = valid_terms[: self.max_features]

        self.vocab_ = {term: idx for idx, (term, _) in enumerate(valid_terms)}

        # Compute smoothed IDF: log((N+1)/(df+1)) + 1
        N = float(self.n_docs_)
        idf_vals = np.zeros(len(self.vocab_))
        for term, idx in self.vocab_.items():
            df = float(doc_freq[term])
            idf_vals[idx] = math.log((N + 1.0) / (df + 1.0)) + 1.0
        self.idf_ = idf_vals
        return self

    def transform(self, corpus: list[str]) -> np.ndarray:
        """Transform documents into L2-normalised TF-IDF matrices.

        Computes raw TF counts, multiplies by IDF weights, then L2-normalises
        each row so documents of different lengths are comparable.

        Args:
            corpus: List of raw text strings.

        Returns:
            TF-IDF matrix, shape (n_docs, vocab_size), L2-normalised rows.
        """
        V = len(self.vocab_)
        X = np.zeros((len(corpus), V), dtype=np.float64)
        for i, text in enumerate(corpus):
            tokens = tokenize(text)
            for tok in tokens:
                if tok in self.vocab_:
                    X[i, self.vocab_[tok]] += 1.0
            # Multiply by IDF
            X[i] *= self.idf_

        # L2 normalise each row
        norms = np.linalg.norm(X, axis=1, keepdims=True)
        norms = np.where(norms == 0, 1.0, norms)
        return X / norms

    def fit_transform(self, corpus: list[str]) -> np.ndarray:
        """Fit and transform in one step.

        Args:
            corpus: List of raw text strings.

        Returns:
            TF-IDF matrix, shape (n_docs, vocab_size).
        """
        return self.fit(corpus).transform(corpus)


tfidf = TFIDF(min_df=MIN_WORD_FREQ, max_features=MAX_VOCAB_SIZE)
X_tfidf_train = tfidf.fit_transform(texts_train)
X_tfidf_test  = tfidf.transform(texts_test)

print(f"TF-IDF vocab size: {len(tfidf.vocab_):,}")
print(f"TF-IDF train shape: {X_tfidf_train.shape}")

# Compare BOW vs TF-IDF sparsity
print(f"BOW   sparsity: {(X_bow_train == 0).mean():.2%}")
print(f"TF-IDF sparsity: {(X_tfidf_train == 0).mean():.2%}")

# Top IDF-weighted terms (highest IDF = rarest, most informative)
top_idf_idx = np.argsort(tfidf.idf_)[::-1][:10]
print(f"\nTop-10 terms by IDF weight (rarest / most informative):")
for rank, idx in enumerate(top_idf_idx, 1):
    term = [t for t, i in tfidf.vocab_.items() if i == idx][0]
    print(f"  {rank:2d}. {term:<22}  IDF = {tfidf.idf_[idx]:.4f}")

---
### Multinomial Naive Bayes: Derivation

**Bayes' theorem** (see 01-07):

$$P(c \mid \mathbf{x}) = \frac{P(\mathbf{x} \mid c)\, P(c)}{P(\mathbf{x})} \propto P(\mathbf{x} \mid c)\, P(c)$$

The **"naive"** assumption: given the class $c$, all word counts are
conditionally independent:

$$P(\mathbf{x} \mid c) = \prod_{w \in \mathcal{V}} P(w \mid c)^{x_w}$$

where $x_w$ is the count of word $w$ in document $\mathbf{x}$.

**Maximum likelihood estimates** (with Laplace smoothing $\alpha$):

$$\hat{P}(w \mid c) = \frac{\text{count}(w, c) + \alpha}{\sum_{w'} \text{count}(w', c) + \alpha |\mathcal{V}|}$$

$$\hat{P}(c) = \frac{N_c}{N}$$

where $N_c$ = number of training documents of class $c$ and $N$ = total.

**Prediction** uses log-probabilities to avoid underflow:

$$\hat{c} = \arg\max_c \left[\log P(c) + \sum_{w} x_w \log P(w \mid c)\right]$$

This is $O(V)$ per document — extremely fast even for large vocabularies.

---
## Part 2 — MultinomialNaiveBayes: Putting It All Together

We build a complete `MultinomialNaiveBayes` class that:
1. Estimates class priors $\log P(c)$ from training label frequencies.
2. Estimates Laplace-smoothed word log-likelihoods $\log P(w \mid c)$ from
   per-class word count matrices.
3. Predicts via the log-sum classifier.

We also implement `GaussianNaiveBayes` for continuous features (used on
TF-IDF in evaluation), demonstrating how the same Bayesian framework
applies to different likelihood assumptions.

In [ ]:
class MultinomialNaiveBayes:
    """Multinomial Naive Bayes classifier for discrete count features.

    Models each feature as a multinomial distribution over vocabulary words.
    Uses Laplace (additive) smoothing to avoid zero probabilities for words
    unseen in a class during training.

    Designed for bag-of-words count matrices where features are non-negative
    integers (or floats treated as counts). Does NOT support negative features.

    Attributes:
        alpha:       Laplace smoothing coefficient (default 1.0).
        log_priors_: Log class prior probabilities, shape (n_classes,).
        log_likelihoods_: Log word likelihoods per class, shape (n_classes, V).
        classes_:    Sorted array of unique class labels.
        n_classes_:  Number of classes.
    """

    def __init__(self, alpha: float = 1.0) -> None:
        """Initialise MultinomialNaiveBayes.

        Args:
            alpha: Laplace smoothing coefficient. alpha=1.0 is standard
                   Laplace smoothing; alpha=0 gives no smoothing (MLE only,
                   risks zero probabilities for unseen words).
        """
        self.alpha = alpha
        self.log_priors_: np.ndarray | None = None
        self.log_likelihoods_: np.ndarray | None = None
        self.classes_: np.ndarray | None = None
        self.n_classes_: int = 0

    def fit(self, X: np.ndarray, y: np.ndarray) -> "MultinomialNaiveBayes":
        """Fit the classifier by estimating priors and Laplace-smoothed likelihoods.

        Args:
            X: Count matrix (non-negative), shape (n_docs, n_features).
            y: Class label array (integers), shape (n_docs,).

        Returns:
            self (fitted classifier).
        """
        self.classes_ = np.unique(y)
        self.n_classes_ = len(self.classes_)
        n_docs, n_features = X.shape

        log_priors = np.zeros(self.n_classes_)
        log_likelihoods = np.zeros((self.n_classes_, n_features))

        for k, cls in enumerate(self.classes_):
            mask = y == cls
            n_cls = int(mask.sum())
            log_priors[k] = math.log(n_cls / n_docs)

            # Sum word counts over all documents in class c
            class_word_counts = X[mask].sum(axis=0)    # shape (n_features,)
            total_count = class_word_counts.sum()

            # Laplace-smoothed log-likelihood: log((count + alpha) / (total + alpha*V))
            smoothed = class_word_counts + self.alpha
            smoothed_total = float(total_count + self.alpha * n_features)
            log_likelihoods[k] = np.log(smoothed / smoothed_total)

        self.log_priors_ = log_priors
        self.log_likelihoods_ = log_likelihoods
        return self

    def predict_log_proba(self, X: np.ndarray) -> np.ndarray:
        """Compute unnormalised log-posterior for each class.

        Returns log P(c) + sum_w x_w * log P(w|c) for each class c.
        Not a true log-probability (unnormalised), but sufficient for argmax.

        Args:
            X: Count matrix, shape (n_docs, n_features).

        Returns:
            Log-score matrix, shape (n_docs, n_classes).

        Raises:
            RuntimeError: If fit() has not been called.
        """
        if self.log_priors_ is None:
            raise RuntimeError("Call fit() before predict_log_proba().")
        # (n_docs, n_features) @ (n_features, n_classes) + (n_classes,)
        return X @ self.log_likelihoods_.T + self.log_priors_

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        """Return normalised posterior probabilities for each class.

        Uses the log-sum-exp trick (see 01-10) to avoid numerical underflow
        when exponentiating large negative log-scores.

        Args:
            X: Count matrix, shape (n_docs, n_features).

        Returns:
            Probability matrix, shape (n_docs, n_classes), rows sum to 1.
        """
        log_scores = self.predict_log_proba(X)           # (n_docs, n_classes)
        # log-sum-exp trick: subtract row max before exp — see Module 01-10
        log_scores -= log_scores.max(axis=1, keepdims=True)
        scores = np.exp(log_scores)
        return scores / scores.sum(axis=1, keepdims=True)

    def predict(self, X: np.ndarray) -> np.ndarray:
        """Predict the most likely class for each document.

        Args:
            X: Count matrix, shape (n_docs, n_features).

        Returns:
            Predicted class label array, shape (n_docs,).
        """
        return self.classes_[np.argmax(self.predict_log_proba(X), axis=1)]

    def score(self, X: np.ndarray, y: np.ndarray) -> float:
        """Return classification accuracy on the given data.

        Args:
            X: Count matrix, shape (n_docs, n_features).
            y: True class labels, shape (n_docs,).

        Returns:
            Accuracy as a float in [0, 1].
        """
        return float(np.mean(self.predict(X) == y))

    def top_features_per_class(
        self,
        class_idx: int,
        vocab: dict[str, int],
        n: int = 15,
    ) -> list[tuple[str, float]]:
        """Return the top-n most discriminative terms for a given class.

        Ranks vocabulary terms by their log-likelihood under this class.
        High log-likelihood terms are words the model associates most
        strongly with this class.

        Args:
            class_idx: Index into self.classes_ (not the raw class label).
            vocab:     Term-to-index mapping from the vectoriser.
            n:         Number of top terms to return.

        Returns:
            List of (term, log_likelihood) tuples, sorted descending.
        """
        log_liks = self.log_likelihoods_[class_idx]
        top_idx  = np.argsort(log_liks)[::-1][:n]
        idx_to_term = {v: k for k, v in vocab.items()}
        return [(idx_to_term[i], float(log_liks[i])) for i in top_idx]


# ── Sanity check on a small toy corpus ───────────────────────────────────────
toy_docs   = ["cat sat mat", "dog runs fast", "cat dog mat", "fast dog cat",
              "space rocket orbit", "nasa orbit moon"]
toy_labels = np.array([0, 1, 0, 1, 2, 2])
toy_bow    = BagOfWords(min_df=1)
X_toy      = toy_bow.fit_transform(toy_docs)

mnb_toy = MultinomialNaiveBayes(alpha=1.0)
mnb_toy.fit(X_toy, toy_labels)
toy_preds = mnb_toy.predict(X_toy)
print(f"Toy corpus train accuracy: {np.mean(toy_preds == toy_labels):.4f}")
print(f"Predicted labels: {toy_preds.tolist()}")
print(f"True labels:      {toy_labels.tolist()}")

# Test prediction on a new document
test_doc = toy_bow.transform(["cat sits on mat"])
proba = mnb_toy.predict_proba(test_doc)
print(f"\nP(class 0 | 'cat sits on mat') = {proba[0, 0]:.4f}  (expected ~high)")
print(f"P(class 1 | 'cat sits on mat') = {proba[0, 1]:.4f}")
print(f"P(class 2 | 'cat sits on mat') = {proba[0, 2]:.4f}")

In [ ]:
class GaussianNaiveBayes:
    """Gaussian Naive Bayes classifier for continuous features.

    Models each feature as a Gaussian (normal) distribution with class-specific
    mean and variance. Unlike MultinomialNB, it handles negative and
    non-count features (e.g., TF-IDF, z-scores).

    Attributes:
        var_smoothing: Small constant added to variance to avoid division by
                       zero for features with zero variance.
        classes_:     Sorted unique class labels.
        log_priors_:  Log class priors, shape (n_classes,).
        mu_:          Per-class feature means, shape (n_classes, n_features).
        sigma2_:      Per-class feature variances, shape (n_classes, n_features).
    """

    def __init__(self, var_smoothing: float = 1e-9) -> None:
        """Initialise GaussianNaiveBayes.

        Args:
            var_smoothing: Variance floor added to all class variances for
                           numerical stability. sklearn default is 1e-9.
        """
        self.var_smoothing = var_smoothing
        self.classes_: np.ndarray | None = None
        self.log_priors_: np.ndarray | None = None
        self.mu_: np.ndarray | None = None
        self.sigma2_: np.ndarray | None = None

    def fit(self, X: np.ndarray, y: np.ndarray) -> "GaussianNaiveBayes":
        """Estimate class priors and per-class Gaussian parameters.

        Args:
            X: Feature matrix (any real values), shape (n_docs, n_features).
            y: Class labels, shape (n_docs,).

        Returns:
            self (fitted classifier).
        """
        self.classes_ = np.unique(y)
        n_docs = len(y)
        n_feat = X.shape[1]
        n_cls  = len(self.classes_)

        self.log_priors_ = np.zeros(n_cls)
        self.mu_         = np.zeros((n_cls, n_feat))
        self.sigma2_     = np.zeros((n_cls, n_feat))

        for k, cls in enumerate(self.classes_):
            mask = y == cls
            self.log_priors_[k] = math.log(float(mask.sum()) / n_docs)
            self.mu_[k]         = X[mask].mean(axis=0)
            self.sigma2_[k]     = X[mask].var(axis=0) + self.var_smoothing

        return self

    def predict_log_proba(self, X: np.ndarray) -> np.ndarray:
        """Compute unnormalised log-posterior using Gaussian likelihoods.

        For each class c and document x:
            log P(c) + sum_j [ -0.5*log(2*pi*sigma2_cj)
                                - (x_j - mu_cj)^2 / (2*sigma2_cj) ]

        Args:
            X: Feature matrix, shape (n_docs, n_features).

        Returns:
            Log-score matrix, shape (n_docs, n_classes).

        Raises:
            RuntimeError: If fit() has not been called.
        """
        if self.mu_ is None:
            raise RuntimeError("Call fit() before predict_log_proba().")
        n_docs = X.shape[0]
        n_cls  = len(self.classes_)
        log_scores = np.zeros((n_docs, n_cls))

        for k in range(n_cls):
            log_scores[:, k] = self.log_priors_[k]
            # Gaussian log-likelihood for each feature (vectorised over docs)
            log_scores[:, k] += -0.5 * np.sum(
                np.log(2 * math.pi * self.sigma2_[k]) +
                (X - self.mu_[k]) ** 2 / self.sigma2_[k],
                axis=1,
            )
        return log_scores

    def predict(self, X: np.ndarray) -> np.ndarray:
        """Predict the most likely class label for each document.

        Args:
            X: Feature matrix, shape (n_docs, n_features).

        Returns:
            Predicted class label array, shape (n_docs,).
        """
        return self.classes_[np.argmax(self.predict_log_proba(X), axis=1)]

    def score(self, X: np.ndarray, y: np.ndarray) -> float:
        """Return classification accuracy.

        Args:
            X: Feature matrix, shape (n_docs, n_features).
            y: True labels, shape (n_docs,).

        Returns:
            Accuracy as a float in [0, 1].
        """
        return float(np.mean(self.predict(X) == y))

---
## Part 3 — Training & Application on 20 Newsgroups

We train four configurations on the full 20 Newsgroups training set and
evaluate on the held-out test set:

| Config | Vectoriser | Classifier |
|--------|-----------|------------|
| A | BOW (scratch) | MultinomialNB (scratch) |
| B | TF-IDF (scratch) | MultinomialNB (scratch) |
| C | BOW (scratch) | GaussianNB (scratch) |
| D | BOW (sklearn) | MultinomialNB (sklearn) |

In [ ]:
# ── Config A: BOW + MultinomialNB (scratch) ───────────────────────────────────
t0 = time.perf_counter()
mnb_bow = MultinomialNaiveBayes(alpha=LAPLACE_ALPHA)
mnb_bow.fit(X_bow_train, y_train_all)
t_train_a = time.perf_counter() - t0

t0 = time.perf_counter()
acc_a = mnb_bow.score(X_bow_test, y_test_all)
t_pred_a = time.perf_counter() - t0

print(f"A (BOW + MultNB scratch)   train={t_train_a*1000:.1f}ms  pred={t_pred_a*1000:.1f}ms  acc={acc_a:.4f}")

# ── Config B: TF-IDF + MultinomialNB (scratch) ────────────────────────────────
# Note: MultinomialNB needs non-negative inputs. TF-IDF is non-negative.
t0 = time.perf_counter()
mnb_tfidf = MultinomialNaiveBayes(alpha=LAPLACE_ALPHA)
mnb_tfidf.fit(X_tfidf_train, y_train_all)
t_train_b = time.perf_counter() - t0

t0 = time.perf_counter()
acc_b = mnb_tfidf.score(X_tfidf_test, y_test_all)
t_pred_b = time.perf_counter() - t0

print(f"B (TF-IDF + MultNB scratch) train={t_train_b*1000:.1f}ms  pred={t_pred_b*1000:.1f}ms  acc={acc_b:.4f}")

# ── Config C: BOW + GaussianNB (scratch) ──────────────────────────────────────
# GaussianNB on BOW — shows that Gaussian assumption is inappropriate for
# count data (violates non-negativity and skewness of word counts)
t0 = time.perf_counter()
gnb = GaussianNaiveBayes()
gnb.fit(X_bow_train, y_train_all)
t_train_c = time.perf_counter() - t0

t0 = time.perf_counter()
acc_c = gnb.score(X_bow_test, y_test_all)
t_pred_c = time.perf_counter() - t0

print(f"C (BOW + GaussNB scratch)  train={t_train_c*1000:.1f}ms  pred={t_pred_c*1000:.1f}ms  acc={acc_c:.4f}")

# ── Config D: sklearn CountVectorizer + sklearn MultinomialNB ─────────────────
# For verification — should match Config A closely
sk_cv  = CountVectorizer(min_df=MIN_WORD_FREQ, max_features=MAX_VOCAB_SIZE,
                         token_pattern=r"[a-z0-9]{2,}", lowercase=True)
X_sk_train = sk_cv.fit_transform(texts_train).toarray()
X_sk_test  = sk_cv.transform(texts_test).toarray()

t0 = time.perf_counter()
sk_mnb = MultinomialNB(alpha=LAPLACE_ALPHA)
sk_mnb.fit(X_sk_train, y_train_all)
t_train_d = time.perf_counter() - t0

t0 = time.perf_counter()
acc_d = sk_mnb.score(X_sk_test, y_test_all)
t_pred_d = time.perf_counter() - t0

print(f"D (BOW + MultNB sklearn)   train={t_train_d*1000:.1f}ms  pred={t_pred_d*1000:.1f}ms  acc={acc_d:.4f}")

print(f"\nScratch vs sklearn gap: {abs(acc_a - acc_d):.4f} (should be <0.02)")
print(f"TF-IDF vs BOW improvement: {acc_b - acc_a:+.4f}")
print(f"Gaussian vs Multinomial:   {acc_c - acc_a:+.4f} (negative = Gaussian worse for counts)")

In [ ]:
# ── Laplace smoothing ablation: vary alpha from 0.001 to 100 ─────────────────
alphas = [0.001, 0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 100.0]
alpha_accs: list[float] = []

for a in alphas:
    clf_a = MultinomialNaiveBayes(alpha=a)
    clf_a.fit(X_bow_train, y_train_all)
    alpha_accs.append(clf_a.score(X_bow_test, y_test_all))

best_alpha = alphas[int(np.argmax(alpha_accs))]
best_alpha_acc = max(alpha_accs)

fig_alpha, ax_alpha = plt.subplots(figsize=(8, 4))
ax_alpha.semilogx(alphas, alpha_accs, marker="o", linewidth=2,
                   color="steelblue", markersize=7)
ax_alpha.axvline(best_alpha, color="firebrick", linestyle="--",
                  label=f"Best alpha={best_alpha}")
ax_alpha.set_xlabel("Laplace smoothing alpha (log scale)")
ax_alpha.set_ylabel("Test accuracy")
ax_alpha.set_title("MultinomialNB: Effect of Laplace Smoothing on Accuracy")
ax_alpha.legend(); ax_alpha.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f"Best alpha: {best_alpha}  |  accuracy: {best_alpha_acc:.4f}")
print(f"alpha=0.001 (near-MLE): {alpha_accs[0]:.4f}")
print(f"alpha=100.0 (heavy smooth): {alpha_accs[-1]:.4f}")
print("\nInsight: Very small alpha risks zero-probability for unseen words.")
print("Very large alpha makes all words equally likely (ignores the data).")

---
### Naive Bayes Variants

**Bernoulli Naive Bayes** models binary presence/absence of words (not counts):

$$P(\mathbf{x} \mid c) = \prod_{w \in \mathcal{V}} P(w \mid c)^{x_w} (1 - P(w \mid c))^{1-x_w}$$

where $x_w \in \{0, 1\}$ indicates whether word $w$ appears at all.
This is sometimes better for short documents where term frequency is noisy.

**Complement Naive Bayes** trains each class model on the *complement* (all
other classes), then negates the score. It corrects for the violation of the
independence assumption by modelling "what this class is *not*":

$$\hat{c} = \arg\min_c \sum_w x_w \log P(w \mid \bar{c})$$

CNB often outperforms standard MultinomialNB on imbalanced text datasets.

In [ ]:
class BernoulliNaiveBayes:
    """Bernoulli Naive Bayes for binary (presence/absence) feature vectors.

    Instead of modelling word counts, models binary indicators x_w in {0,1}.
    For each class c, estimates P(w|c) = presence probability of word w.
    The likelihood accounts for BOTH presence and absence of each word,
    unlike MultinomialNB which only considers present words.

    Attributes:
        alpha:      Laplace smoothing coefficient.
        classes_:   Sorted unique class labels.
        log_priors_: Log class prior probabilities, shape (n_classes,).
        log_p1_:    Log P(w=1 | c) for each class, shape (n_classes, V).
        log_p0_:    Log P(w=0 | c) = log(1 - P(w=1|c)), shape (n_classes, V).
    """

    def __init__(self, alpha: float = 1.0) -> None:
        """Initialise BernoulliNaiveBayes.

        Args:
            alpha: Laplace smoothing coefficient.
        """
        self.alpha = alpha
        self.classes_: np.ndarray | None = None
        self.log_priors_: np.ndarray | None = None
        self.log_p1_: np.ndarray | None = None
        self.log_p0_: np.ndarray | None = None

    def fit(self, X: np.ndarray, y: np.ndarray) -> "BernoulliNaiveBayes":
        """Fit Bernoulli NB from a binary feature matrix.

        Args:
            X: Binary feature matrix (0/1 or binarised counts), shape (n, V).
            y: Class labels, shape (n,).

        Returns:
            self (fitted classifier).
        """
        self.classes_ = np.unique(y)
        n_docs = len(y)
        n_feat = X.shape[1]
        n_cls  = len(self.classes_)
        X_bin  = (X > 0).astype(float)      # binarise in case counts passed

        self.log_priors_ = np.zeros(n_cls)
        self.log_p1_     = np.zeros((n_cls, n_feat))
        self.log_p0_     = np.zeros((n_cls, n_feat))

        for k, cls in enumerate(self.classes_):
            mask = y == cls
            n_cls_docs = float(mask.sum())
            self.log_priors_[k] = math.log(n_cls_docs / n_docs)

            # P(w=1|c) = (count of docs in c containing w + alpha) /
            #            (n_cls_docs + 2*alpha)
            word_presence = X_bin[mask].sum(axis=0)
            p1 = (word_presence + self.alpha) / (n_cls_docs + 2.0 * self.alpha)
            self.log_p1_[k] = np.log(p1)
            self.log_p0_[k] = np.log(1.0 - p1)

        return self

    def predict_log_proba(self, X: np.ndarray) -> np.ndarray:
        """Compute unnormalised log-posterior scores.

        Bernoulli likelihood: sum_w [ x_w * log P(w=1|c) + (1-x_w) * log P(w=0|c) ]

        Args:
            X: Feature matrix (binarised), shape (n_docs, V).

        Returns:
            Log-score matrix, shape (n_docs, n_classes).

        Raises:
            RuntimeError: If fit() has not been called.
        """
        if self.log_p1_ is None:
            raise RuntimeError("Call fit() before predict_log_proba().")
        X_bin = (X > 0).astype(float)
        # log P(c) + x * log_p1 + (1-x) * log_p0
        # = log P(c) + x * (log_p1 - log_p0) + sum(log_p0)
        # This factorisation is efficient: only one matrix multiply
        diff    = self.log_p1_ - self.log_p0_           # (n_cls, V)
        log_p0_sum = self.log_p0_.sum(axis=1)           # (n_cls,)
        scores  = X_bin @ diff.T + log_p0_sum + self.log_priors_
        return scores

    def predict(self, X: np.ndarray) -> np.ndarray:
        """Predict most likely class for each document.

        Args:
            X: Feature matrix, shape (n_docs, V).

        Returns:
            Predicted class label array, shape (n_docs,).
        """
        return self.classes_[np.argmax(self.predict_log_proba(X), axis=1)]

    def score(self, X: np.ndarray, y: np.ndarray) -> float:
        """Return classification accuracy.

        Args:
            X: Feature matrix, shape (n_docs, V).
            y: True labels, shape (n_docs,).

        Returns:
            Accuracy as a float in [0, 1].
        """
        return float(np.mean(self.predict(X) == y))


class ComplementNaiveBayes:
    """Complement Naive Bayes — models the complement of each class.

    For each class c, trains the model on all documents NOT in c,
    then classifies by choosing the class with the minimum complement score.
    CNB corrects for class imbalance and often outperforms standard
    MultinomialNB on multi-class text tasks.

    Reference: Rennie et al. (2003) "Tackling the Poor Assumptions of Naive
    Bayes Text Classifiers", ICML.

    Attributes:
        alpha:          Laplace smoothing coefficient.
        classes_:       Sorted unique class labels.
        log_priors_:    Log class prior probabilities, shape (n_classes,).
        log_comp_liks_: Log complement likelihoods, shape (n_classes, V).
    """

    def __init__(self, alpha: float = 1.0) -> None:
        """Initialise ComplementNaiveBayes.

        Args:
            alpha: Laplace smoothing coefficient.
        """
        self.alpha = alpha
        self.classes_: np.ndarray | None = None
        self.log_priors_: np.ndarray | None = None
        self.log_comp_liks_: np.ndarray | None = None

    def fit(self, X: np.ndarray, y: np.ndarray) -> "ComplementNaiveBayes":
        """Estimate complement class log-likelihoods.

        For each class c, pools all documents in the complement (y != c)
        and estimates smoothed word probabilities from those documents.

        Args:
            X: Count matrix, shape (n_docs, V).
            y: Class labels, shape (n_docs,).

        Returns:
            self (fitted classifier).
        """
        self.classes_ = np.unique(y)
        n_docs = len(y)
        n_feat = X.shape[1]
        n_cls  = len(self.classes_)

        self.log_priors_    = np.zeros(n_cls)
        self.log_comp_liks_ = np.zeros((n_cls, n_feat))

        for k, cls in enumerate(self.classes_):
            mask = y == cls
            self.log_priors_[k] = math.log(float(mask.sum()) / n_docs)

            # Complement: documents NOT in this class
            comp_mask = ~mask
            comp_word_counts = X[comp_mask].sum(axis=0)
            comp_total = float(comp_word_counts.sum())
            smoothed = comp_word_counts + self.alpha
            smoothed_total = comp_total + self.alpha * n_feat
            self.log_comp_liks_[k] = np.log(smoothed / smoothed_total)

        # Weight normalisation (optional but recommended by Rennie et al.)
        # Normalise each row so |log_comp_liks| sums to 1
        abs_sum = np.abs(self.log_comp_liks_).sum(axis=1, keepdims=True)
        self.log_comp_liks_ /= np.where(abs_sum == 0, 1.0, abs_sum)

        return self

    def predict_log_proba(self, X: np.ndarray) -> np.ndarray:
        """Compute complement scores (negated for argmax convention).

        CNB classifies by MINIMISING the complement score.
        We negate it so that argmax gives the correct prediction.

        Args:
            X: Count matrix, shape (n_docs, V).

        Returns:
            Negated complement log-score matrix, shape (n_docs, n_classes).

        Raises:
            RuntimeError: If fit() has not been called.
        """
        if self.log_comp_liks_ is None:
            raise RuntimeError("Call fit() before predict_log_proba().")
        # Complement score: log P(c) + x @ log_comp_lik
        # Negate so argmax = argmin of complement
        comp_scores = X @ self.log_comp_liks_.T + self.log_priors_
        return -comp_scores

    def predict(self, X: np.ndarray) -> np.ndarray:
        """Predict most likely class for each document.

        Args:
            X: Count matrix, shape (n_docs, V).

        Returns:
            Predicted class label array, shape (n_docs,).
        """
        return self.classes_[np.argmax(self.predict_log_proba(X), axis=1)]

    def score(self, X: np.ndarray, y: np.ndarray) -> float:
        """Return classification accuracy.

        Args:
            X: Count matrix, shape (n_docs, V).
            y: True labels, shape (n_docs,).

        Returns:
            Accuracy as a float in [0, 1].
        """
        return float(np.mean(self.predict(X) == y))


# ── Train and compare all NB variants ─────────────────────────────────────────
print("Naive Bayes Variant Comparison on 20 Newsgroups (6 classes):")
print(f"{'Variant':<28} {'Train (ms)':>12} {'Test Acc':>10}")
print("-" * 55)

variants: dict[str, object] = {
    "MultinomialNB (scratch)":  MultinomialNaiveBayes(alpha=best_alpha),
    "BernoulliNB (scratch)":    BernoulliNaiveBayes(alpha=best_alpha),
    "ComplementNB (scratch)":   ComplementNaiveBayes(alpha=best_alpha),
    "MultinomialNB (sklearn)":  MultinomialNB(alpha=best_alpha),
}

variant_accs: dict[str, float] = {}
for name, clf in variants.items():
    t0 = time.perf_counter()
    clf.fit(X_bow_train, y_train_all)
    t_tr = time.perf_counter() - t0
    acc = clf.score(X_bow_test, y_test_all)
    variant_accs[name] = acc
    print(f"{name:<28} {t_tr*1000:>10.1f}ms {acc:>10.4f}")

# Bar chart
fig_var, ax_var = plt.subplots(figsize=(10, 4))
names_v = list(variant_accs.keys())
accs_v  = list(variant_accs.values())
bar_colors = ["steelblue", "darkorange", "firebrick", "purple"]
bars_v = ax_var.bar(names_v, accs_v, color=bar_colors, alpha=0.85,
                    edgecolor="black", linewidth=0.8)
for bar, val in zip(bars_v, accs_v):
    ax_var.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
                f"{val:.4f}", ha="center", va="bottom", fontsize=9)
ax_var.set_ylim(min(accs_v) - 0.05, 1.01)
ax_var.set_ylabel("Test accuracy")
ax_var.set_title(f"Naive Bayes Variants (BOW, alpha={best_alpha}, 6 newsgroups)")
ax_var.tick_params(axis="x", labelrotation=15)
ax_var.grid(True, alpha=0.3, axis="y")
plt.tight_layout(); plt.show()

---
### Learning Curve: How Much Data Does Naive Bayes Need?

One of NB's practical strengths is that it needs very little training data
to achieve near-optimal performance. We measure accuracy as we grow the
training set from 5% to 100%, revealing how quickly NB converges.

In [ ]:
def learning_curve_nb(
    X_tr: np.ndarray,
    y_tr: np.ndarray,
    X_te: np.ndarray,
    y_te: np.ndarray,
    fractions: list[float],
    alpha: float = 1.0,
    seed: int = SEED,
) -> tuple[list[int], list[float], list[float]]:
    """Compute MultinomialNB learning curve by varying training set size.

    For each fraction f, randomly samples f * n_train documents and
    trains a MultinomialNB on them, measuring test accuracy. Repeats
    three times and averages to reduce variance from sampling.

    Args:
        X_tr:      Full training count matrix, shape (n_train, V).
        y_tr:      Full training labels, shape (n_train,).
        X_te:      Test count matrix, shape (n_test, V).
        y_te:      Test labels, shape (n_test,).
        fractions: List of fractions of training data to use (0 < f <= 1).
        alpha:     Laplace smoothing coefficient.
        seed:      Random seed for subsampling.

    Returns:
        Tuple (n_samples_list, mean_accs, std_accs).
            n_samples_list: Number of training samples for each fraction.
            mean_accs:      Mean test accuracy over 3 trials.
            std_accs:       Standard deviation of test accuracy over 3 trials.
    """
    rng_lc = np.random.default_rng(seed)
    n_train = len(y_tr)
    n_samples_list: list[int] = []
    mean_accs: list[float] = []
    std_accs: list[float] = []

    for frac in fractions:
        n_use = max(int(frac * n_train), len(np.unique(y_tr)))
        trial_accs: list[float] = []
        for _ in range(3):
            idx = rng_lc.choice(n_train, size=n_use, replace=False)
            clf_lc = MultinomialNaiveBayes(alpha=alpha)
            clf_lc.fit(X_tr[idx], y_tr[idx])
            trial_accs.append(clf_lc.score(X_te, y_te))
        n_samples_list.append(n_use)
        mean_accs.append(float(np.mean(trial_accs)))
        std_accs.append(float(np.std(trial_accs)))

    return n_samples_list, mean_accs, std_accs


fractions = [0.02, 0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50, 0.60, 0.75, 0.90, 1.00]
n_list, mean_accs, std_accs = learning_curve_nb(
    X_bow_train, y_train_all,
    X_bow_test, y_test_all,
    fractions=fractions,
    alpha=best_alpha,
)

fig_lc, ax_lc = plt.subplots(figsize=(9, 4))
mean_arr = np.array(mean_accs)
std_arr  = np.array(std_accs)
ax_lc.plot(n_list, mean_arr, marker="o", linewidth=2, color="steelblue",
            markersize=6, label="Mean test accuracy")
ax_lc.fill_between(n_list, mean_arr - std_arr, mean_arr + std_arr,
                    alpha=0.25, color="steelblue", label="+/- 1 std")
ax_lc.set_xlabel("Training set size (documents)")
ax_lc.set_ylabel("Test accuracy")
ax_lc.set_title("MultinomialNB Learning Curve (BOW, 6 Newsgroups)")
ax_lc.legend(); ax_lc.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print("Learning curve summary:")
print(f"  2% ({n_list[0]} docs)  -> acc = {mean_accs[0]:.4f}")
print(f" 10% ({n_list[2]} docs)  -> acc = {mean_accs[2]:.4f}")
print(f" 50% ({n_list[7]} docs)  -> acc = {mean_accs[7]:.4f}")
print(f"100% ({n_list[-1]} docs) -> acc = {mean_accs[-1]:.4f}")
print("\nNaive Bayes converges quickly — even 10% of training data often")
print("achieves 80-90% of the full-data accuracy. This makes NB ideal")
print("for low-resource text classification scenarios.")

---
## Part 4 — Evaluation & Analysis

We evaluate the best configuration with a classification report, confusion
matrix, per-class top features, conditional independence analysis, and error
inspection.

---
### Vocabulary Analysis: OOV Rate & Log-Likelihood Ratio

Two practical questions when deploying a Naive Bayes text classifier:
1. **Out-of-vocabulary (OOV) rate** — what fraction of test words are absent
   from the training vocabulary? These are silently ignored at inference time.
2. **Most discriminative features** — which terms have the highest *log-
   likelihood ratio* between their most-likely class and the overall average?
   These terms give the clearest class signal.

In [ ]:
def oov_analysis(
    bow_vectoriser: "BagOfWords",
    train_corpus: list[str],
    test_corpus: list[str],
) -> dict[str, float]:
    """Compute out-of-vocabulary statistics between train and test corpora.

    Counts what fraction of unique test words are unseen in the training
    vocabulary and what fraction of total test word occurrences are OOV.
    Both metrics are important: the latter shows how much of the test
    content is lost when unknown words are ignored.

    Args:
        bow_vectoriser: Fitted BagOfWords with vocab_ attribute.
        train_corpus:   List of training text strings.
        test_corpus:    List of test text strings.

    Returns:
        Dictionary with keys:
            "train_vocab_size": number of terms in training vocab.
            "test_unique_words": number of unique words in test set.
            "oov_unique_fraction": fraction of unique test words that are OOV.
            "oov_token_fraction":  fraction of total test tokens that are OOV.
    """
    vocab = set(bow_vectoriser.vocab_.keys())

    # Collect all test words
    test_word_counts: Counter = Counter()
    for text in test_corpus:
        for tok in tokenize(text):
            test_word_counts[tok] += 1

    total_test_tokens = sum(test_word_counts.values())
    unique_test_words = len(test_word_counts)

    oov_unique  = sum(1 for w in test_word_counts if w not in vocab)
    oov_tokens  = sum(cnt for w, cnt in test_word_counts.items() if w not in vocab)

    return {
        "train_vocab_size":     len(vocab),
        "test_unique_words":    unique_test_words,
        "oov_unique_fraction":  oov_unique / max(unique_test_words, 1),
        "oov_token_fraction":   oov_tokens / max(total_test_tokens, 1),
    }


def log_likelihood_ratio_features(
    clf: "MultinomialNaiveBayes",
    vocab: dict[str, int],
    n_top: int = 10,
) -> list[tuple[str, int, float]]:
    """Find the most discriminative terms via log-likelihood ratio.

    For each term w, computes:
        LLR(w) = max_c log P(w|c) - mean_c log P(w|c)
    A high LLR means the term is highly specific to one class compared to
    the average, making it maximally discriminative.

    Args:
        clf:   Fitted MultinomialNaiveBayes with log_likelihoods_ attribute.
        vocab: Term-to-column-index mapping.
        n_top: Number of top terms to return.

    Returns:
        List of (term, best_class_idx, llr_score) tuples, sorted descending.
    """
    log_liks = clf.log_likelihoods_             # (n_classes, V)
    mean_liks = log_liks.mean(axis=0)           # (V,)
    max_liks  = log_liks.max(axis=0)            # (V,)
    llr       = max_liks - mean_liks            # (V,)
    best_cls  = log_liks.argmax(axis=0)         # (V,)

    top_idx = np.argsort(llr)[::-1][:n_top]
    idx_to_term = {v: k for k, v in vocab.items()}
    return [(idx_to_term[i], int(best_cls[i]), float(llr[i])) for i in top_idx]


# OOV analysis
oov_stats = oov_analysis(bow, texts_train, texts_test)
print("Vocabulary & OOV Analysis:")
print(f"  Training vocab size   : {oov_stats['train_vocab_size']:>8,}")
print(f"  Unique test words     : {oov_stats['test_unique_words']:>8,}")
print(f"  OOV unique fraction   : {oov_stats['oov_unique_fraction']:.2%}")
print(f"  OOV token fraction    : {oov_stats['oov_token_fraction']:.2%}")
print("  (OOV words are silently ignored at inference time)")

# Log-likelihood ratio: most discriminative terms
llr_terms = log_likelihood_ratio_features(mnb_best, bow.vocab_, n_top=20)
print("\nTop-20 most discriminative terms (log-likelihood ratio):")
print(f"  {'Term':<22} {'Best class':<20} {'LLR':>8}")
print(f"  {'-'*55}")
for term, cls_i, llr in llr_terms[:20]:
    cls_name = class_names[cls_i].split(".")[-1]
    print(f"  {term:<22} {cls_name:<20} {llr:>8.4f}")

# Visualise LLR bar chart for top 15
fig_llr, ax_llr = plt.subplots(figsize=(10, 5))
top15 = llr_terms[:15]
terms15 = [t for t, _, _ in top15]
llr15   = [s for _, _, s in top15]
cls15   = [c for _, c, _ in top15]
colors15 = ["steelblue", "darkorange", "firebrick", "purple", "green", "brown"]
bar_cols = [colors15[c % len(colors15)] for c in cls15]
ax_llr.barh(range(15), llr15[::-1], color=bar_cols[::-1], alpha=0.85,
            edgecolor="black", linewidth=0.5)
ax_llr.set_yticks(range(15))
ax_llr.set_yticklabels(terms15[::-1], fontsize=9)
ax_llr.set_xlabel("Log-Likelihood Ratio (max - mean over classes)")
ax_llr.set_title("Top-15 Most Discriminative Terms (MultinomialNB)")
ax_llr.grid(True, alpha=0.3, axis="x")
plt.tight_layout(); plt.show()

---
### Naive Bayes vs Logistic Regression: Speed & Accuracy

A classic comparison in text classification: **Naive Bayes** (generative,
closed-form) vs **Logistic Regression** (discriminative, iterative).
Both handle high-dimensional sparse text features well, but have very
different training dynamics.

In [ ]:
from sklearn.linear_model import LogisticRegression

# Compare on same BOW features for fairness
comparison_models: dict[str, object] = {
    "MultinomialNB (scratch)": MultinomialNaiveBayes(alpha=best_alpha),
    "BernoulliNB (scratch)":   BernoulliNaiveBayes(alpha=best_alpha),
    "ComplementNB (scratch)":  ComplementNaiveBayes(alpha=best_alpha),
    "MultinomialNB (sklearn)": MultinomialNB(alpha=best_alpha),
    "Logistic Reg. (L2)":      LogisticRegression(C=1.0, max_iter=1000,
                                                   random_state=SEED,
                                                   solver="lbfgs",
                                                   multi_class="multinomial"),
    "Logistic Reg. (L1)":      LogisticRegression(C=1.0, max_iter=1000,
                                                   random_state=SEED,
                                                   solver="saga",
                                                   penalty="l1"),
}

print(f"{'Model':<30} {'Train (ms)':>12} {'Pred (ms)':>10} {'Acc':>8}")
print("-" * 65)

comp_accs: dict[str, float] = {}
comp_train_t: dict[str, float] = {}

for name, clf in comparison_models.items():
    t0 = time.perf_counter()
    clf.fit(X_bow_train, y_train_all)
    t_tr = time.perf_counter() - t0

    t0 = time.perf_counter()
    preds = clf.predict(X_bow_test)
    t_pr = time.perf_counter() - t0

    acc = float(np.mean(preds == y_test_all))
    comp_accs[name]    = acc
    comp_train_t[name] = t_tr * 1000
    print(f"{name:<30} {t_tr*1000:>10.1f}ms {t_pr*1000:>9.1f}ms {acc:>8.4f}")

print()
best_nb_acc = max(comp_accs[k] for k in comp_accs if "NB" in k or "Bayes" in k)
best_lr_acc = max(comp_accs[k] for k in comp_accs if "Logistic" in k)
print(f"Best NB accuracy : {best_nb_acc:.4f}")
print(f"Best LR accuracy : {best_lr_acc:.4f}")
print(f"LR - NB gap      : {best_lr_acc - best_nb_acc:+.4f}")
nb_fastest_t = min(comp_train_t[k] for k in comp_train_t if "scratch" in k.lower() or "sklearn" in k.lower())
lr_fastest_t = min(comp_train_t[k] for k in comp_train_t if "Logistic" in k)
print(f"\nTraining speed ratio (LR / best-NB): {lr_fastest_t / nb_fastest_t:.1f}x slower")
print("Insight: NB trains in O(N*V) with a single pass; LR requires iterative")
print("optimisation (L-BFGS / SAGA), which is 10-50x slower on large corpora.")

# ── Accuracy vs training-time scatter plot for all models ─────────────────────
fig_scat, ax_scat = plt.subplots(figsize=(9, 5))

marker_map = {
    "MultinomialNB (scratch)": ("o", "steelblue"),
    "BernoulliNB (scratch)":   ("s", "darkorange"),
    "ComplementNB (scratch)":  ("^", "firebrick"),
    "MultinomialNB (sklearn)": ("D", "purple"),
    "Logistic Reg. (L2)":      ("P", "green"),
    "Logistic Reg. (L1)":      ("*", "brown"),
}

for name, (marker, color) in marker_map.items():
    ax_scat.scatter(
        comp_train_t[name],
        comp_accs[name],
        marker=marker,
        color=color,
        s=120,
        label=name,
        zorder=3,
        edgecolors="black",
        linewidths=0.8,
    )

ax_scat.set_xlabel("Training time (ms)")
ax_scat.set_ylabel("Test accuracy")
ax_scat.set_title("Accuracy vs Training Time: NB vs Logistic Regression")
ax_scat.legend(fontsize=8, loc="lower right")
ax_scat.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ── Practical guide: when to choose NB vs LR ──────────────────────────────────
print()
print("When to choose Naive Bayes vs Logistic Regression:")
print("  Use NB when:")
print("    - Training data is very limited (NB converges faster)")
print("    - Inference latency matters (NB prediction is O(n_classes * V))")
print("    - You need a probabilistic output (calibration is poor but ordinal ranks hold)")
print("    - Streaming / online learning is required (NB updates incrementally)")
print("  Use LR when:")
print("    - Training data is abundant (LR catches up and often surpasses NB)")
print("    - Features are correlated (LR handles this better than NB)")
print("    - You need good probability calibration (LR is better calibrated)")
print("    - L1 regularisation for feature selection is desired")

In [ ]:
# ── Use best-alpha model for all subsequent analysis ──────────────────────────
mnb_best = MultinomialNaiveBayes(alpha=best_alpha)
mnb_best.fit(X_bow_train, y_train_all)
y_pred_best = mnb_best.predict(X_bow_test)

print("Classification Report (BOW + MultinomialNB, best alpha):")
print("=" * 70)
report = classification_report(
    y_test_all, y_pred_best,
    target_names=[name.split(".")[-1] for name in class_names],
    digits=4,
)
print(report)

# Per-class accuracy
print("Per-class accuracy:")
for cls_idx, name in enumerate(class_names):
    mask = y_test_all == cls_idx
    per_acc = float(np.mean(y_pred_best[mask] == y_test_all[mask]))
    print(f"  {name:<30} {per_acc:.4f}  ({mask.sum()} samples)")

In [ ]:
cm = confusion_matrix(y_test_all, y_pred_best)
short_class_names = [name.split(".")[-1] for name in class_names]

fig_cm, ax_cm = plt.subplots(figsize=(8, 7))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=short_class_names)
disp.plot(ax=ax_cm, cmap="Blues", colorbar=True, values_format="d")
ax_cm.set_title(f"Confusion Matrix\nMultinomialNB (BOW, alpha={best_alpha})")
ax_cm.tick_params(axis="x", labelrotation=30)
plt.tight_layout(); plt.show()

# Most confused pairs
off_diag = [(cm[i, j], class_names[i], class_names[j])
            for i in range(len(class_names)) for j in range(len(class_names)) if i != j]
off_diag.sort(reverse=True)
print("Top-5 most confused category pairs (true -> predicted):")
for count, true_cls, pred_cls in off_diag[:5]:
    true_short = true_cls.split(".")[-1]
    pred_short = pred_cls.split(".")[-1]
    print(f"  {true_short:<15} -> {pred_short:<15}: {count} errors")

In [ ]:
# ── Most informative features (terms) per class ───────────────────────────────
N_TERMS = 12
fig_feat, axes_feat = plt.subplots(2, 3, figsize=(15, 8))
axes_feat = axes_feat.flatten()

for cls_idx, (ax_f, cls_name) in enumerate(zip(axes_feat, class_names)):
    top_terms = mnb_best.top_features_per_class(cls_idx, bow.vocab_, n=N_TERMS)
    terms  = [t for t, _ in top_terms]
    scores = [s for _, s in top_terms]
    # Convert log-likelihood to probability for display
    probs  = [math.exp(s) * 100 for s in scores]

    ax_f.barh(range(N_TERMS), probs[::-1], color="steelblue", alpha=0.85,
              edgecolor="black", linewidth=0.5)
    ax_f.set_yticks(range(N_TERMS))
    ax_f.set_yticklabels(terms[::-1], fontsize=8)
    ax_f.set_xlabel("P(word | class) %", fontsize=8)
    short_name = cls_name.replace("talk.", "").replace("comp.", "").replace("rec.", "")
    ax_f.set_title(f"[{cls_idx}] {short_name}", fontsize=9)
    ax_f.grid(True, alpha=0.3, axis="x")

plt.suptitle("Top 12 Most Likely Words per Class (MultinomialNB)", fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# ── When does conditional independence break? ─────────────────────────────────
# Compute pairwise correlation between high-TF-IDF words within a class.
# High correlations indicate word pairs that co-occur more than independently
# predicted — the places where the "naive" assumption is most violated.

def pairwise_correlation_top_words(
    X: np.ndarray,
    y: np.ndarray,
    class_idx: int,
    vocab: dict[str, int],
    top_n: int = 30,
) -> tuple[np.ndarray, list[str]]:
    """Compute pairwise Pearson correlation among top-frequency words in a class.

    Selects the top_n most frequent words for a given class, extracts their
    count columns from the document matrix, and returns the correlation matrix.

    Args:
        X:         Count matrix, shape (n_docs, V).
        y:         Class labels, shape (n_docs,).
        class_idx: Target class index (integer label).
        vocab:     Term-to-column-index mapping.
        top_n:     Number of top words to include.

    Returns:
        Tuple (corr_matrix, term_list).
            corr_matrix: Pearson correlation matrix, shape (top_n, top_n).
            term_list:   List of top_n term strings.
    """
    mask = y == class_idx
    X_cls = X[mask]
    col_sums = X_cls.sum(axis=0)
    top_cols = np.argsort(col_sums)[::-1][:top_n]
    idx_to_term = {v: k for k, v in vocab.items()}
    term_list = [idx_to_term[c] for c in top_cols]

    X_sub = X_cls[:, top_cols].astype(float)
    # Pearson correlation (numpy corrcoef is column-wise, so transpose)
    corr = np.corrcoef(X_sub.T)
    return corr, term_list


# Show correlation heatmap for sci.space (class 1)
corr_space, terms_space = pairwise_correlation_top_words(
    X_bow_train, y_train_all,
    class_idx=1,   # sci.space
    vocab=bow.vocab_,
    top_n=20,
)

fig_corr, ax_corr = plt.subplots(figsize=(9, 8))
im_corr = ax_corr.imshow(corr_space, cmap="coolwarm", vmin=-0.5, vmax=0.5)
ax_corr.set_xticks(range(20)); ax_corr.set_yticks(range(20))
ax_corr.set_xticklabels(terms_space, rotation=60, ha="right", fontsize=7)
ax_corr.set_yticklabels(terms_space, fontsize=7)
plt.colorbar(im_corr, ax=ax_corr, fraction=0.046, pad=0.04)
ax_corr.set_title("Word Co-occurrence Correlation (sci.space)\n"
                   "High correlation = conditional independence violated here")
plt.tight_layout(); plt.show()

# Quantify: fraction of highly correlated word pairs
upper_tri = corr_space[np.triu_indices(20, k=1)]
high_corr = float((np.abs(upper_tri) > 0.3).mean())
print(f"Fraction of word pairs with |correlation| > 0.3 in sci.space: {high_corr:.1%}")
print("This illustrates where the 'naive' independence assumption fails:")
print("domain-specific words co-occur far more than independent models predict.")
print("Yet MultinomialNB still achieves good accuracy because the posterior")
print("*direction* (argmax) is often correct even when probabilities are wrong.")

In [ ]:
# ── Error analysis: examine misclassified documents ───────────────────────────
errors_idx = np.where(y_pred_best != y_test_all)[0]
total_errors = len(errors_idx)
total_test   = len(y_test_all)
print(f"Total errors: {total_errors} / {total_test}  ({total_errors/total_test:.1%} error rate)")
print(f"Accuracy    : {1 - total_errors/total_test:.4f}")

# Show a few misclassified examples with model confidence
print()
print("--- Misclassified examples ---")
proba_all = mnb_best.predict_proba(X_bow_test)

shown = 0
for err_i in errors_idx[:20]:
    true_lbl  = y_test_all[err_i]
    pred_lbl  = y_pred_best[err_i]
    true_name = class_names[true_lbl].split(".")[-1]
    pred_name = class_names[pred_lbl].split(".")[-1]
    confidence = proba_all[err_i, pred_lbl]

    # Only show high-confidence errors (model is confidently wrong)
    if confidence > 0.6:
        doc_snippet = texts_test[err_i][:180].replace("\n", " ")
        print(f"True: {true_name:<15}  Pred: {pred_name:<15}  Conf: {confidence:.4f}")
        print(f"  Text: {doc_snippet!r}")
        print()
        shown += 1
        if shown >= 4:
            break

# Confidence histogram: correct vs incorrect predictions
conf_correct   = [proba_all[i, y_pred_best[i]]
                  for i in range(len(y_test_all)) if y_pred_best[i] == y_test_all[i]]
conf_incorrect = [proba_all[i, y_pred_best[i]]
                  for i in range(len(y_test_all)) if y_pred_best[i] != y_test_all[i]]

fig_conf, ax_conf = plt.subplots(figsize=(8, 4))
ax_conf.hist(conf_correct,   bins=30, alpha=0.7, color="steelblue",
             label=f"Correct ({len(conf_correct)})", density=True)
ax_conf.hist(conf_incorrect, bins=30, alpha=0.7, color="firebrick",
             label=f"Incorrect ({len(conf_incorrect)})", density=True)
ax_conf.set_xlabel("Model confidence (predicted class probability)")
ax_conf.set_ylabel("Density")
ax_conf.set_title("Confidence Distribution: Correct vs Incorrect Predictions")
ax_conf.legend(); ax_conf.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

mean_conf_correct   = float(np.mean(conf_correct))
mean_conf_incorrect = float(np.mean(conf_incorrect))
print(f"Mean confidence (correct)  : {mean_conf_correct:.4f}")
print(f"Mean confidence (incorrect): {mean_conf_incorrect:.4f}")
print("MultinomialNB tends to be overconfident -- a known limitation of the")
print("naive independence assumption (calibration is poor; accuracy is good).")

---
## Part 5 — Summary & Lessons Learned

### Key Takeaways

- **Naive Bayes is a probabilistic discriminator:** by applying Bayes'
  theorem with a conditional independence assumption, it converts word counts
  into class probabilities via a single pass over the data — O(N · V) training.
- **Laplace smoothing is essential:** without it, a single unseen word
  in a test document assigns zero probability to an entire class, breaking
  the argmax. Smoothing regularises the word frequency estimates.
- **TF-IDF down-weights common words**, but MultinomialNB learns to ignore
  them automatically via low class-conditional log-likelihoods — so the
  accuracy improvement from TF-IDF is often modest.
- **The "naive" assumption is almost always violated** (words co-occur),
  yet NB still gets the *argmax* class right in most cases because the
  *ordering* of posterior scores is more robust than the absolute values.
  This is why NB is overconfident (poor calibration) but accurate.
- **BOW and TF-IDF** are the precursors to word embeddings (Module 7) and
  subword tokenisation (Module 8). They convert arbitrary text into fixed-
  length numerical vectors that any downstream model can consume.

### What's Next
→ **02-09 (Stacking & Voting Ensembles)** combines multiple classifiers
(including Naive Bayes) into meta-learner stacking and voting, showing when
diversity among base models improves combined accuracy.